# 3. QLoRA fine-tuning and ablations
This trains the judge with `scripts/train_qlora.py` (4-bit NF4 base + LoRA adapters) on the five-step reasoning targets. It picks the best epoch on validation F1 and evaluates once on the test split.

The ablations are:
- verdict-only vs reasoning;
- QLoRA vs 16-bit LoRA;
- the target-language data curve (H3).

**Runtime:** T4 GPU. Run the quick check first.

In [ ]:
REPO_URL = "https://github.com/KingQuinn/cross-lingual-groundedness-judge.git"
import os
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!git pull -q
!pip -q install -r requirements.txt

In [ ]:
# Everything that costs API calls or GPU time is saved to Google Drive, so a
# disconnect never loses work; re-running a cell resumes where it stopped.
from google.colab import drive, userdata
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cross-lingual-judge"
os.makedirs(WORK, exist_ok=True)

In [ ]:
# Tiny Aya and Llama are gated: accept each licence on its Hugging Face page
# (logged in), then add a Hugging Face token as the Colab secret HF_TOKEN.
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
if not os.path.exists("data/items.jsonl"):
    !mkdir -p data && cp $WORK/data/items.jsonl data/
PREDS = f"{WORK}/preds"
os.makedirs(PREDS, exist_ok=True)
LIMIT = 500   # about 500 original test items, sampled as whole documents (identical for every judge); None = all
LIM = f"--limit {LIMIT}" if LIMIT else ""
MODELS = ["CohereLabs/tiny-aya-earth", "Qwen/Qwen3-4B", "meta-llama/Llama-3.2-3B-Instruct"]
CKPT = f"{WORK}/checkpoints"
BASE = "CohereLabs/tiny-aya-earth"   # choose from the zero-shot results

## Quick run: 200 items, 1 epoch. Check peak memory, time and that the loss falls.

In [ ]:
!python scripts/train_qlora.py --model $BASE --max_train 200 --epochs 1 --out /content/quick
s = json.load(open("/content/quick/train_stats.json"))
print(f"{s['train_minutes']:.1f} min, peak {s['peak_memory_gb']:.2f} GB, trainable {s['trainable_params']:,}")
print([round(h["loss"], 3) for h in s["log_history"] if "loss" in h])

## Train, select the epoch on validation, test once

In [ ]:
import glob, sys
sys.path.insert(0, ".")
from judge.metrics import prf

def train_select_test(run, extra="", mode="reasoning", eval_flags=""):
    !python scripts/train_qlora.py --model $BASE --epochs 2 --mode $mode $extra --out $CKPT/$run
    best = None
    for ck in sorted(glob.glob(f"{CKPT}/{run}/checkpoint-*")):
        name = f"val-{run}-{os.path.basename(ck)}"
        !python scripts/run_judge.py --judge hf:$BASE --adapter $ck --mode $mode $eval_flags --split val --limit 200 --no_variants --name $name --out_dir /content/val
        f1 = prf([json.loads(l) for l in open(f"/content/val/{name}.jsonl")])["f1"]
        print(ck, "val F1", round(f1, 3))
        if best is None or f1 > best[1]:
            best = (ck, f1)
    !python scripts/run_judge.py --judge hf:$BASE --adapter {best[0]} --mode $mode $eval_flags $LIM --name $run --out_dir $PREDS
    return best

In [ ]:
print(train_select_test("aya-earth-qlora"))   # H2: main fine-tuned judge

## Ablations (run as time allows; each records time and peak memory)

In [ ]:
print(train_select_test("aya-earth-verdict", mode="verdict"))                     # what does the reasoning buy?
print(train_select_test("aya-earth-en-only", extra="--train_langs en"))           # H3: English only
print(train_select_test("aya-earth-t100", extra="--target_n 100"))                # H3: +100 per language
print(train_select_test("aya-earth-t500", extra="--target_n 500"))                # H3: +500 per language
print(train_select_test("aya-earth-lora16", extra="--no_4bit", eval_flags="--no_4bit"))  # QLoRA vs 16-bit LoRA

## Analysis → README

In [ ]:
!rm -rf results/preds && mkdir -p results && cp -r $PREDS results/preds && cp $WORK/fertility.json results/ 2>/dev/null
!mkdir -p results/train_stats && for d in $CKPT/*/; do cp $d/train_stats.json results/train_stats/$(basename $d).json 2>/dev/null; done
!python scripts/analyze.py
from IPython.display import Markdown, display
display(Markdown(open("results/RESULTS.md").read()))